# Fine tune MAE-DFER on ElderReact dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/sunlicai/MAE-DFER.git
%cd MAE-DFER
!pip install timm==0.4.12 einops decord tensorboardX opencv-python scikit-learn

Mounted at /content/drive
Cloning into 'MAE-DFER'...
remote: Enumerating objects: 259, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 259 (delta 50), reused 17 (delta 17), pack-reused 181 (from 1)
Receiving objects: 100% (259/259), 16.52 MiB | 14.73 MiB/s, done.
Resolving deltas: 100% (98/98), done.
/content/MAE-DFER
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 36.2 MB/s  0:00:00
  Attempting uninstall: timm
    Found existing installation: timm 1.0.31
    Uninstalling timm-1.0.31:
      Successfully uninstalled timm-1.0.31
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [timm]


In [ ]:
# Fine-tune MAE-DFER (pre-trained checkpoint-49) on ElderReact (multi-label, 6 emotions)
#
# Colab usage:
#   from google.colab import drive; drive.mount('/content/drive')
#   !git clone https://github.com/sunlicai/MAE-DFER.git /content/MAE-DFER
#   !pip install timm==0.4.12 einops decord tensorboardX opencv-python scikit-learn
#   !python finetune_elderreact.py

import os, sys, random, time, glob
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from decord import VideoReader, cpu
from sklearn.metrics import f1_score, average_precision_score

# ============================ CONFIG ============================
REPO_DIR   = '/content/MAE-DFER'
CKPT       = '/content/drive/MyDrive/plugg/year5/projektkurs/checkpoints/checkpoint-49.pth'     # <-- edit
DATA_DIR   = '/content/drive/MyDrive/plugg/year5/projektkurs/ElderReact_Data'               # <-- edit (folder with videos)
LABEL_FILE = '/content/drive/MyDrive/plugg/year5/projektkurs/ElderReact_Data/train_labels.txt'    # <-- edit (name, 6 binary cols)
OUT_DIR    = '/content/drive/MyDrive/plugg/year5/projektkurs/elderreact_ft'

MODEL_NAME = 'vit_base_dim512_no_depth_patch16_160'  # check against the repo's DFEW fine-tune script
EMOTIONS   = ['anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise']  # set to match your column order
NUM_FRAMES, TUBELET, SIZE = 16, 2, 160

EPOCHS, BATCH_SIZE = 30, 8
LR_BACKBONE, LR_HEAD, WEIGHT_DECAY = 5e-5, 5e-4, 0.05
WARMUP_EPOCHS = 3
VAL_FRACTION, SEED = 0.2, 42     # random split (swap for the official ElderReact split when you have it)
MAX_VIDEOS = 10                # e.g. 200 to start tiny
USE_FACE_CROP = True             # Haar-cascade face crop (fast, rough); False = full frame
NUM_WORKERS = 2
# =================================================================

sys.path.append(REPO_DIR)
import modeling_finetune  # noqa: F401  (registers models with timm)
from timm.models import create_model

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs(OUT_DIR, exist_ok=True)

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
VIDEO_EXTS = ['', '.mp4', '.avi', '.mov', '.mkv', '.webm']


# ------------------------- labels / data -------------------------
def find_video(name):
    for ext in VIDEO_EXTS:
        p = os.path.join(DATA_DIR, name + ext)
        if os.path.isfile(p):
            return p
    hits = glob.glob(os.path.join(DATA_DIR, '**', os.path.splitext(name)[0] + '.*'), recursive=True)
    return hits[0] if hits else None


def read_labels(path):
    items, missing = [], 0
    with open(path) as f:
        for line in f:
            parts = line.replace(',', ' ').split()
            if len(parts) < 7:
                continue
            try:
                y = [float(v) for v in parts[1:7]]
            except ValueError:      # header line
                continue
            vp = find_video(parts[0])
            if vp is None:
                missing += 1
                continue
            items.append((vp, y))
    print(f'Found {len(items)} labelled videos ({missing} listed in txt but not on disk)')
    return items


# _face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
USE_FACE_CROP = False

def face_box(frame, margin=0.3):
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    faces = _face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(40, 40))
    if len(faces) == 0:
        return None
    x, y, w, h = max(faces, key=lambda b: b[2] * b[3])
    H, W = frame.shape[:2]
    s = int(max(w, h) * (1 + margin)); cx, cy = x + w // 2, y + h // 2
    x0, y0 = max(0, cx - s // 2), max(0, cy - s // 2)
    return x0, y0, min(W, x0 + s), min(H, y0 + s)


class ElderReactDataset(Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train

    def __len__(self):
        return len(self.items)

    def _indices(self, n):
        seg = np.linspace(0, n, NUM_FRAMES + 1)
        if self.train:   # random frame inside each segment
            idx = [random.randint(int(seg[i]), max(int(seg[i]), int(seg[i + 1]) - 1)) for i in range(NUM_FRAMES)]
        else:            # centre of each segment
            idx = [int((seg[i] + seg[i + 1]) / 2) for i in range(NUM_FRAMES)]
        return np.clip(idx, 0, n - 1)

    def __getitem__(self, i):
        path, y = self.items[i]
        try:
            vr = VideoReader(path, ctx=cpu(0))
            frames = vr.get_batch(self._indices(len(vr))).asnumpy()          # T,H,W,3
        except Exception as e:
            print('bad video', path, e)
            frames = np.zeros((NUM_FRAMES, SIZE, SIZE, 3), dtype=np.uint8)

        if USE_FACE_CROP:
            box = face_box(frames[len(frames) // 2])
            if box:
                x0, y0, x1, y1 = box
                frames = frames[:, y0:y1, x0:x1]

        flip = self.train and random.random() < 0.5
        out = []
        for f in frames:
            f = cv2.resize(f, (SIZE, SIZE))
            if flip:
                f = f[:, ::-1]
            out.append((f.astype(np.float32) / 255.0 - MEAN) / STD)
        x = torch.from_numpy(np.stack(out).transpose(3, 0, 1, 2).copy())    # C,T,H,W
        return x, torch.tensor(y, dtype=torch.float32)


# ------------------------- model -------------------------
def build_model():
    model = create_model(MODEL_NAME, pretrained=False, num_classes=len(EMOTIONS),
                         all_frames=NUM_FRAMES, tubelet_size=TUBELET, drop_path_rate=0.1)
    ckpt = torch.load(CKPT, map_location='cpu', weights_only=False)
    sd = next((ckpt[k] for k in ('model', 'module', 'state_dict') if k in ckpt), ckpt)
    sd = {(k[len('encoder.'):] if k.startswith('encoder.') else k): v for k, v in sd.items()}
    sd = {k: v for k, v in sd.items() if not k.startswith('decoder') and not k.startswith('head')}
    msg = model.load_state_dict(sd, strict=False)
    print('missing keys   :', msg.missing_keys)
    print('unexpected keys:', msg.unexpected_keys)
    return model


# ------------------------- train / eval -------------------------
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    probs, ys, loss_sum = [], [], 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type='cuda', enabled=device == 'cuda'):
            logits = model(x)
        loss_sum += criterion(logits.float(), y).item() * len(x)
        probs.append(torch.sigmoid(logits.float()).cpu().numpy()); ys.append(y.cpu().numpy())
    p, y = np.concatenate(probs), np.concatenate(ys)
    pred = (p > 0.5).astype(int)
    f1s = [f1_score(y[:, c], pred[:, c], zero_division=0) for c in range(y.shape[1])]
    aps = [average_precision_score(y[:, c], p[:, c]) if y[:, c].sum() > 0 else np.nan for c in range(y.shape[1])]
    acc = (pred == y).mean(0)
    return dict(loss=loss_sum / len(y), f1=f1s, mean_f1=float(np.mean(f1s)),
                mAP=float(np.nanmean(aps)), acc=acc)


def main():
    items = read_labels(LABEL_FILE)
    if MAX_VIDEOS:
        random.shuffle(items); items = items[:MAX_VIDEOS]
    random.shuffle(items)
    n_val = max(1, int(len(items) * VAL_FRACTION))
    val_items, train_items = items[:n_val], items[n_val:]
    print(f'train {len(train_items)} | val {len(val_items)}')

    tr = DataLoader(ElderReactDataset(train_items, True), batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=NUM_WORKERS, drop_last=True, pin_memory=True)
    va = DataLoader(ElderReactDataset(val_items, False), batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)

    # class imbalance -> pos_weight
    Y = np.array([y for _, y in train_items])
    pos = Y.sum(0); neg = len(Y) - pos
    pos_weight = torch.tensor(np.clip(neg / np.maximum(pos, 1), 1, 10), dtype=torch.float32).to(device)
    print('positives per class:', dict(zip(EMOTIONS, pos.astype(int))), '| pos_weight:', pos_weight.cpu().numpy().round(2))
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    model = build_model().to(device)

    head_params = [p for n, p in model.named_parameters() if n.startswith('head') or n.startswith('fc_norm')]
    head_ids = {id(p) for p in head_params}
    backbone = [p for p in model.parameters() if id(p) not in head_ids]
    optimizer = torch.optim.AdamW([{'params': backbone, 'lr': LR_BACKBONE},
                                   {'params': head_params, 'lr': LR_HEAD}], weight_decay=WEIGHT_DECAY)
    base_lrs = [LR_BACKBONE, LR_HEAD]
    total_steps, warm_steps = EPOCHS * len(tr), WARMUP_EPOCHS * len(tr)
    scaler = torch.cuda.amp.GradScaler(enabled=device == 'cuda')

    best, step = -1.0, 0
    for epoch in range(EPOCHS):
        model.train(); t0 = time.time(); run = 0.0
        for x, y in tr:
            # warmup + cosine schedule
            scale = (step + 1) / warm_steps if step < warm_steps else \
                0.5 * (1 + np.cos(np.pi * (step - warm_steps) / max(1, total_steps - warm_steps)))
            for g, b in zip(optimizer.param_groups, base_lrs):
                g['lr'] = b * scale
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type='cuda', enabled=device == 'cuda'):
                logits = model(x)
            loss = criterion(logits.float(), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            run += loss.item(); step += 1

        m = evaluate(model, va, criterion)
        print(f'epoch {epoch+1:02d}/{EPOCHS} | train loss {run/len(tr):.4f} | val loss {m["loss"]:.4f} '
              f'| mean F1 {m["mean_f1"]:.3f} | mAP {m["mAP"]:.3f} | {time.time()-t0:.0f}s')
        print('   per-class F1:', {e: round(f, 3) for e, f in zip(EMOTIONS, m['f1'])})
        if m['mean_f1'] > best:
            best = m['mean_f1']
            torch.save({'model': model.state_dict(), 'epoch': epoch, 'metrics': m['mean_f1']},
                       os.path.join(OUT_DIR, 'best.pth'))
            print('   saved best.pth')
    print('best val mean F1:', round(best, 3))


if __name__ == '__main__':
    main()

Found 615 labelled videos (0 listed in txt but not on disk)
train 492 | val 123
positives per class: {'anger': np.int64(152), 'disgust': np.int64(120), 'fear': np.int64(73), 'happiness': np.int64(261), 'sadness': np.int64(97), 'surprise': np.int64(230)} | pos_weight: [2.24 3.1  5.74 1.   4.07 1.14]
missing keys   : ['blocks.0.norm1.weight', 'blocks.0.norm1.bias', 'blocks.0.attn.q_bias', 'blocks.0.attn.v_bias', 'blocks.0.attn.qkv.weight', 'blocks.0.attn.proj.weight', 'blocks.0.attn.proj.bias', 'blocks.1.norm1.weight', 'blocks.1.norm1.bias', 'blocks.1.attn.q_bias', 'blocks.1.attn.v_bias', 'blocks.1.attn.qkv.weight', 'blocks.1.attn.proj.weight', 'blocks.1.attn.proj.bias', 'blocks.2.norm1.weight', 'blocks.2.norm1.bias', 'blocks.2.attn.q_bias', 'blocks.2.attn.v_bias', 'blocks.2.attn.qkv.weight', 'blocks.2.attn.proj.weight', 'blocks.2.attn.proj.bias', 'blocks.3.norm1.weight', 'blocks.3.norm1.bias', 'blocks.3.attn.q_bias', 'blocks.3.attn.v_bias', 'blocks.3.attn.qkv.weight', 'blocks.3.attn.pro

/tmp/ipykernel_2567/2833618000.py:201: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=device == 'cuda')
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
